# Week 2 Homework: Data Preprocessing of Machine Learning Solution Project Dataset (Submission)

    Data preprocessing ensures that the data used to develop machine learning solutions is clean, consistent, and relevant. For the next two weeks, you will preprocess the FinMark Corporation datasets. The main objective of this activity is to exercise your familiarity and skills in data preprocessing using Python.
    
    To carry out this activity, follow the instructions below:

        1. Access the company website. Review the problem statement and download the provided datasets.
        2. Load the downloaded datasets in Jupyter Notebooks and preprocess them using Pandas:
           a. Check for duplicate rows and remove them.
           b. Identify columns with missing values and decide how to handle them.
           c. Ensure columns have the appropriate data types.
        4. Once completed, save the preprocessed datasets and prepare to discuss the output with your team.

    Problem Statement: FinMark Corporation has collected data such as customer transactions, social media, and demographics, but they struggle to find useful insights due to data overload and inconsistencies. As a result, the company lacks a clear understanding of market trends and customer behavior. 
    
    Project Deliverables: As their data analyst, the company wants you to develop a machine learning solution to address these challenges. Over the next 12 weeks, the company wants you to develop the following:

        1. Exploratory Data Analysis (EDA) - explores the main characteristics of the collected data, identifies key patterns, and detects anomalies in the company datasets.
        2. Data Visualization - discovers relationships between variables and identifies patterns and trends in the datasets.
        3. Presentation of Machine Learning Solution Project - a presentation of your development process and findings.

WORKFLOW:
1. REMOVED EXACT DUPLICATES

2. DECIDED HOW TO HANDLE RECORDS WITH DUPLICATE UNIQUE IDS
    2.1. Observations: CustomerID duplicates (46 records) - either duplicate records have the same Age but different SignupDate (24 records) OR different Age but same SignupDate  (22 records)
    Decision:
        2.1.1 For duplicate records that have the same Age but different SignupDate (24 records), I retained the one with the more recent SignupDate as it is more likely up-to-date.
        2.1.2 For duplicate records that have different Age but same SignupDate (22 records), I picked the more reasonable age and deleted the records with either -1, Unknown, or 150 value.
   
    2.2. Observations: TransactionID duplicates (30 records) - either duplicate records have the same TransactionDate but different Amount (16 records), different TransactionDate but same Amount (8 records), and varying TransactionDate and Amount (6 records)
    Decision:
        2.2.1 For duplicate records that have the same TransactionDate but different Amount (16 records):
            a. Positive Amount vs "Free" - retained record with positive Amount since there was a PaymentMethod indicated
            b. Positive Amount vs Zero - retained record with positive Amount since there was a PaymentMethod indicated
            c. Positive Amount vs -100 - deducted the -100 against positive Amount since this could possibly be a voucher, discount, or a refund; provided that the positive Amount is bigger than 100.
        2.2.2 For duplicate records that have different TransactionDate but same Amount (10 records), I retained the record with the more recent TransactionDate.
        2.2.3 For duplicate records that have varying TransactionDate and Amount (4 records), I deleted both records since it was hard to determine which record is more reliable.
    2.3. Observations: InteractionID duplicates (40 records) - either duplicate records have the same InteractionDate but different Sentiment (18 records) or different InteractionDate but same Sentiment (22 records).
    Decision:
        2.3.1 For duplicate records that have different InteractionDate but same Sentiment (26 records), I retained the records with the more recent InteractionDate.
        2.3.2 For duplicate records that have the same InteractionDate but different Sentiment (14 records), I decided to delete them both since there's no way to determine which record is more reliable.

3. HANDLED ANOMALOUS RECORDS
    3.1 Customer Demographics
        3.1.1 'Unknown' Age - changed to null
        3.1.2 Anomalous Age (-1,150) - retained and flagged as outliers
    3.2 Customer Transactions
        3.2.1 'Free' Amount - changed to 0.00
        3.2.2 Anomalous Amount (-100) - retained and flagged as outliers

4. HANDLED NULL VALUES
    4.1 Identified and deleted records that have more than 1 missing value.
    4.2 Performed mean imputation for numeric columns, while mode imputation for the categorical columns.

5. STANDARDIZED FORMAT


# Importing tools

In [559]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np # linear algebra
import scipy as sp
from scipy import stats
import scipy.cluster.hierarchy as sch
from itertools import combinations  # added for phase 2 deduplication
import regex
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

In [560]:
# Set pandas display options to ensure all columns are displayed in a single line
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)  # Set a large width to avoid line breaks

In [561]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_demographics_contaminated.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_transactions_contaminated.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/social_media_interactions_contaminated.csv')

# Define datasets and dataset_names
datasets = [data1, data2, data3] #replace
dataset_names = ['customer_demographics','customer_transactions','social_media_interactions']
dataset_titles = ['Customer Demographics','Customer Transactions','Social Media Interactions']

# Preprocessing: Identifying shape of the dataset and data types of each columns

In [563]:
print("*************************************************************************")
print("Homework: Data Preprocessing of Machine Learning Solution Project Dataset") #replace
print("*************************************************************************")

# Check for number of columns and rows for each dataset
print("\nPrior to preprocessing:")
for df, dataset_name in zip(datasets, dataset_names):
    print(f"\n{dataset_name} has {df.shape[1]} columns and {df.shape[0]} rows.")
    # Check for data types
    print(f"Data types for each column:\n{df.dtypes}")

*************************************************************************
Homework: Data Preprocessing of Machine Learning Solution Project Dataset
*************************************************************************

Prior to preprocessing:

customer_demographics has 6 columns and 3200 rows.
Data types for each column:
CustomerID     object
Age            object
Gender         object
Location       object
IncomeLevel    object
SignupDate     object
dtype: object

customer_transactions has 6 columns and 3200 rows.
Data types for each column:
CustomerID         object
TransactionID      object
TransactionDate    object
Amount             object
ProductCategory    object
PaymentMethod      object
dtype: object

social_media_interactions has 6 columns and 3200 rows.
Data types for each column:
CustomerID         object
InteractionID      object
InteractionDate    object
Platform           object
InteractionType    object
Sentiment          object
dtype: object


In [564]:
for df, dataset_name in zip(datasets, dataset_names):
    print(f"\n {dataset_name} statistics: \n{df.describe()}")


 customer_demographics statistics: 
                                  CustomerID   Age  Gender      Location IncomeLevel  SignupDate
count                                   3200  2909    3200          3200        2897        3200
unique                                  3000    58       2          2696           3        1544
top     1a5115af-2fa7-4198-b838-be2cdeccb768  26.0  Female  Port Michael        High  2024-06-08
freq                                       2    72    1610             6        1004           9

 customer_transactions statistics: 
                                  CustomerID                         TransactionID TransactionDate Amount ProductCategory PaymentMethod
count                                   3200                                  3200            3200   2896            2901          3200
unique                                  1871                                  3000             808   2603               5             4
top     49f06ea9-4ae5-4ce5-98ce-d

# Display unique values per column

In [566]:
# Function to display unique values per column for a dataset
def display_unique_values(datasets, dataset_names):
    """
    Displays the unique values in each column of the given datasets.
    
    Parameters:
    datasets (list of pd.DataFrame): List of datasets to inspect.
    dataset_names (list of str): List of dataset names corresponding to the dataframes.
    
    Returns:
    None
    """
    for df, dataset_name in zip(datasets, dataset_names):
        print('**********************************************************')
        print(f"--- Unique Values in Columns for {dataset_name} ---")
        print('**********************************************************')
        for column in df.columns:
            unique_values = df[column].unique()
            print(f"Column: {column}")
            print(f"Unique Values ({len(unique_values)}): {unique_values}")
            print('-' * 50)

# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_demographics_contaminated.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_transactions_contaminated.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/social_media_interactions_contaminated.csv')

# Define datasets and dataset names
datasets = [data1, data2, data3]
dataset_names = ['Customer Demographics', 'Customer Transactions', 'Social Media Interactions']

# Call the function to display unique values
display_unique_values(datasets, dataset_names)

**********************************************************
--- Unique Values in Columns for Customer Demographics ---
**********************************************************
Column: CustomerID
Unique Values (3000): ['9207fa75-5758-48d1-94ad-19c041e0520f'
 '5fb09cd8-a473-46f7-80bd-6e49cf509078'
 'c139496e-cc89-498a-bd90-1fb4627b6cff' ...
 '1f0f2c2d-eb0a-4ffb-a21a-684f892bd7fd'
 '9eead230-c70b-473b-a234-7002688b42b6'
 'db9c2206-4487-47e9-8fb0-0ddc81b7310c']
--------------------------------------------------
Column: Age
Unique Values (59): ['51.0' nan '37.0' '44.0' '50.0' '53.0' '150' '59.0' '29.0' '31.0' '24.0'
 '49.0' '65.0' '45.0' '64.0' '52.0' '61.0' '47.0' '25.0' '18.0' '19.0'
 '70.0' '32.0' '23.0' '40.0' '63.0' '54.0' '68.0' '39.0' '33.0' '46.0'
 '58.0' '48.0' '22.0' '28.0' '34.0' '26.0' '36.0' '21.0' '56.0' 'Unknown'
 '62.0' '67.0' '27.0' '55.0' '42.0' '20.0' '35.0' '43.0' '60.0' '57.0'
 '66.0' '69.0' '30.0' '41.0' '-1' '38.0' '-1.0' '150.0']
------------------------------------

# Identifying columns with null values

In [568]:
# Define datasets and dataset_names
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_demographics_contaminated.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_transactions_contaminated.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/social_media_interactions_contaminated.csv')

datasets = [data1, data2, data3]
dataset_names = ['customer_demographics','customer_transactions','social_media_interactions']
dataset_titles = ['Customer Demographics','Customer Transactions','Social Media Interactions']

# Function to check for missing values and print only columns with missing values
def check_missing_values(datasets, dataset_titles):
    """
    Checks for missing values in the datasets and prints the variables with missing values.

    Parameters:
    datasets (list of pd.DataFrame): List of dataframes to check.
    dataset_names (list of str): List of dataset names corresponding to the dataframes.
    """
    for df, dataset_title in zip(datasets, dataset_titles):
        # print(f"\nMissing values for {dataset_name}:")
        missing_values = df.isnull().sum()  # Count missing values in each column
        missing_values = missing_values[missing_values > 0]  # Filter only those with missing values
        
        if not missing_values.empty:
            print(f"\nVariables with missing values in {dataset_title}:")
            for column, count in missing_values.items():
                print(f"{column}: {count} missing values")
        else:
            print(f"\nNo missing values found in {dataset_title}.")
            
# Call the function to check for and print columns with missing values
check_missing_values(datasets, dataset_titles)


Variables with missing values in Customer Demographics:
Age: 291 missing values
IncomeLevel: 303 missing values

Variables with missing values in Customer Transactions:
Amount: 304 missing values
ProductCategory: 299 missing values

Variables with missing values in Social Media Interactions:
Platform: 311 missing values
Sentiment: 329 missing values


# Calculating missingness rate per column

In [570]:
# Define datasets and dataset_names
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_demographics_contaminated.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_transactions_contaminated.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/social_media_interactions_contaminated.csv')

# Define datasets and dataset_names
datasets = [data1, data2, data3]
dataset_names = ['customer_demographics','customer_transactions','social_media_interactions']
dataset_titles = ['Customer Demographics','Customer Transactions','Social Media Interactions']

# Calculate missingness for each column
def missingness_rate(datasets, dataset_titles):
    for df, dataset_title in zip(datasets, dataset_titles):
        missingness = df.isnull().sum() / len(df) * 100
        # Display missingness percentage for each column
        print(f"\nMissingness rate per column in {dataset_title}:\n{missingness}")

missingness_rate(datasets, dataset_titles)


Missingness rate per column in Customer Demographics:
CustomerID     0.00000
Age            9.09375
Gender         0.00000
Location       0.00000
IncomeLevel    9.46875
SignupDate     0.00000
dtype: float64

Missingness rate per column in Customer Transactions:
CustomerID         0.00000
TransactionID      0.00000
TransactionDate    0.00000
Amount             9.50000
ProductCategory    9.34375
PaymentMethod      0.00000
dtype: float64

Missingness rate per column in Social Media Interactions:
CustomerID          0.00000
InteractionID       0.00000
InteractionDate     0.00000
Platform            9.71875
InteractionType     0.00000
Sentiment          10.28125
dtype: float64


# Identifying the number of null values in a record/per column

In [572]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_demographics_contaminated.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_transactions_contaminated.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/social_media_interactions_contaminated.csv')

# Define datasets and dataset_names
datasets = [data1, data2, data3]
dataset_names = ['customer_demographics', 'customer_transactions', 'social_media_interactions']

# Specify columns to search for missing values in each dataset
columns_to_search = {
    'customer_demographics': ['CustomerID','Age', 'Gender','Location','IncomeLevel','SignupDate'],  
    'customer_transactions': ['CustomerID', 'TransactionID','TransactionDate', 'Amount','ProductCategory','PaymentMethod'],  
    'social_media_interactions': ['CustomerID', 'InteractionID','InteractionDate','Platform','InteractionType','Sentiment']
}

def search_null_columns(datasets, dataset_names, columns_to_search):
    for dataset, dataset_name in zip(datasets, dataset_names):
        print('\n********************************************************************************************************')
        print(f"Processing dataset: {dataset_name}")
        print('********************************************************************************************************')
        for column in columns_to_search[dataset_name]:
            # Search for records where the specified column is missing
            record_match = dataset.loc[dataset[column].isnull()].copy()  # Explicitly create a copy
            print('-----------------------------------------------------------------------------------------------')
            if record_match.empty:
                print(f"No missing values found in the column '{column}' in the dataset '{dataset_name}'.")
                continue

            # Count the number of missing values in each row for the matched records
            record_match['missing_count'] = record_match.isnull().sum(axis=1)

            # Display the search results along with the missing value count for each record
            print(f"Search results for records that have missing values under '{column}':")
            print(record_match[[column, 'missing_count']])

            # Find the minimum and maximum number of missing values
            min_missing = record_match['missing_count'].min()
            max_missing = record_match['missing_count'].max()
            print(f"\nMinimum number of missing values in records: {min_missing}")
            print(f"Maximum number of missing values in records: {max_missing}")

            # Group by the number of missing values and count how many rows have the same number of missing values
            missing_value_summary = record_match['missing_count'].value_counts().sort_index()

            # Convert to DataFrame and rename the columns
            missing_value_summary_df = missing_value_summary.reset_index()
            missing_value_summary_df.columns = ['Number of Missing Values', 'Count of Record(s)']

            # Display the grouped and sorted number of missing values and the corresponding count of records
            print("\nSummary of records by the number of missing values:")
            print(missing_value_summary_df)

# Call the function with multiple datasets
search_null_columns(datasets, dataset_names, columns_to_search)



********************************************************************************************************
Processing dataset: customer_demographics
********************************************************************************************************
-----------------------------------------------------------------------------------------------
No missing values found in the column 'CustomerID' in the dataset 'customer_demographics'.
-----------------------------------------------------------------------------------------------
Search results for records that have missing values under 'Age':
      Age  missing_count
1     NaN              1
6     NaN              1
7     NaN              1
8     NaN              2
9     NaN              1
...   ...            ...
3148  NaN              1
3159  NaN              2
3179  NaN              1
3182  NaN              1
3197  NaN              1

[291 rows x 2 columns]

Minimum number of missing values in records: 1
Maximum number of missing v

# a. Check for duplicate rows and remove them.

In [574]:
# DEDUPLICATION PHASE 1
# Phase 1 of Deduplication: Removing exact duplicates
def deduplication_phase1(datasets, dataset_names, dataset_titles):
    """
    Performs the first phase of deduplication on multiple datasets.
    
    Parameters:
    datasets (list of pd.DataFrame): List of dataframes to deduplicate.
    dataset_names (list of str): List of dataset names corresponding to the dataframes.
    dataset_titles (list of str): List of dataset titles for printing in the output.
    
    Returns:
    dict: A dictionary where the keys are cleaned dataset names and the values are the cleaned dataframes.
    """
    deduplicated_datasets_phase1 = {}

    for df, dataset_name, dataset_title in zip(datasets, dataset_names, dataset_titles):
        
        # Check for duplicates
        duplicates_phase1 = df.duplicated()  # This returns a boolean Series    
        
        # Count the number of duplicate rows
        num_duplicates_phase1 = duplicates_phase1.sum()  # Sum of True values gives the count
        
        # Print the number of duplicate rows
        if num_duplicates_phase1 > 0:
            print("\n------------------------------------------------------------------")
            print(f"--- First Phase of Deduplication for {dataset_title} ---")
            print("------------------------------------------------------------------")
            print(f"Findings: {dataset_title} has {num_duplicates_phase1} duplicate rows.")
        else:
            print(f"Findings: {dataset_title} has no duplicate rows.")
        
        # Remove duplicates and keep the first occurrence
        df_deduplication_phase1 = df.drop_duplicates(keep='first')
        
        # Print the number of duplicates removed
        num_removed_phase1 = len(df) - len(df_deduplication_phase1)
        print(f"Action/s: Removed {num_removed_phase1} duplicate rows from {dataset_title}.")
        
        # Print the number of rows after removal
        print(f"Results: {dataset_title} now has {len(df_deduplication_phase1)} rows after the first phase of deduplication.")
        
        # Save cleaned dataset with modified name
        deduplicated_dataset_name = dataset_name + "_deduplication_phase1"
        deduplicated_datasets_phase1[deduplicated_dataset_name] = df_deduplication_phase1
        
        # Save cleaned dataset to CSV
        df_deduplication_phase1.to_csv(f"{deduplicated_dataset_name}.csv", index=False)
        print(f"Cleaned dataset saved as {deduplicated_dataset_name}.csv")

    return deduplicated_datasets_phase1


# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_demographics_contaminated.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/customer_transactions_contaminated.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Raw Dataset/social_media_interactions_contaminated.csv')

# Define datasets and dataset_names
datasets = [data1, data2, data3] #replace
dataset_names = ['customer_demographics','customer_transactions','social_media_interactions']
dataset_titles = ['Customer Demographics','Customer Transactions','Social Media Interactions']

# Call deduplication_phase1 for all datasets
deduplicated_datasets_phase1 = deduplication_phase1(datasets, dataset_names, dataset_titles)



------------------------------------------------------------------
--- First Phase of Deduplication for Customer Demographics ---
------------------------------------------------------------------
Findings: Customer Demographics has 177 duplicate rows.
Action/s: Removed 177 duplicate rows from Customer Demographics.
Results: Customer Demographics now has 3023 rows after the first phase of deduplication.
Cleaned dataset saved as customer_demographics_deduplication_phase1.csv

------------------------------------------------------------------
--- First Phase of Deduplication for Customer Transactions ---
------------------------------------------------------------------
Findings: Customer Transactions has 185 duplicate rows.
Action/s: Removed 185 duplicate rows from Customer Transactions.
Results: Customer Transactions now has 3015 rows after the first phase of deduplication.
Cleaned dataset saved as customer_transactions_deduplication_phase1.csv

---------------------------------------

# Identifying duplicate unique ids

In [576]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_deduplication_phase1.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_deduplication_phase1.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_deduplication_phase1.csv')

# Define datasets and their names and titles
datasets = [data1, data2, data3]
dataset_names = ['customer_demographics', 'customer_transactions', 'social_media_interactions']
dataset_titles = ['Customer Demographics', 'Customer Transactions', 'Social Media Interactions']
unique_id_columns = ['CustomerID', 'TransactionID', 'InteractionID']

# Phase 2 of Deduplication: Count Duplicates in Unique ID and Print Unique Identifiers
def find_duplicate_uid(datasets, dataset_names, dataset_titles, unique_id_columns):
    for df, dataset_name, dataset_title, unique_id_column in zip(datasets, dataset_names, dataset_titles, unique_id_columns):
        print("\n---------------------------------------------------------------")
        print(f"--- Second Phase of Deduplication for {dataset_title} ---")
        print("---------------------------------------------------------------")
        
        # Check for duplicates based on unique identifier
        duplicates_unique_id = df.duplicated(subset=[unique_id_column], keep=False)  
        
        # Count the number of duplicate rows
        num_duplicates_phase2 = duplicates_unique_id.sum()  # Sum of True values gives the count
        
        # Print the number of duplicate rows
        if num_duplicates_phase2 > 0:
            print(f"Findings: {dataset_title} has {num_duplicates_phase2} duplicate rows based on '{unique_id_column}'.")
            
            # Print the unique identifiers of the duplicates
            duplicate_ids = df[duplicates_unique_id][unique_id_column].unique()  # Get unique identifiers
            print(f"Duplicate Unique Identifiers: \n{duplicate_ids}")
        else:
            print(f"Findings: {dataset_title} has no duplicate rows based on '{unique_id_column}'.")

# Run deduplication for all datasets
find_duplicate_uid(datasets, dataset_names, dataset_titles, unique_id_columns)


---------------------------------------------------------------
--- Second Phase of Deduplication for Customer Demographics ---
---------------------------------------------------------------
Findings: Customer Demographics has 46 duplicate rows based on 'CustomerID'.
Duplicate Unique Identifiers: 
['bbf7c36f-3745-4d3a-9e33-f927c2755f65'
 '6b2f8f03-ab9e-48a6-a07e-10e794b04fe0'
 'b72f2a59-aab0-447d-bb31-9bdc1405cc72'
 '1e9e7dcb-e612-46d3-bae6-0c2f47fa0220'
 '48313b18-ccdb-4946-b958-1ebfc92dcc2a'
 '50465bde-6a9a-466d-804a-a8f046809a23'
 '29436bcf-d6f8-44d1-a856-0794abb562ab'
 '835384f5-c976-481d-8961-cc827dc62c6b'
 '890f10ce-36f6-4993-9a2e-a00dfdf01609'
 'ce5cf079-859b-43fa-9609-c6bb7e12c6b7'
 '3814b8e4-cade-4665-86af-dda3e3628f17'
 '76c1d696-0f1c-44b5-b749-8859545658eb'
 'bb07fa4a-8680-4b77-b1ad-5d2f67570252'
 '08995799-8d0f-40d6-986b-452074709b5f'
 'bbe76a26-234f-4a70-b71f-fa9b183f6353'
 '7eb7f7df-9b0a-416a-829e-bcf4d8bd57dc'
 '89ca5faa-aba9-4b14-b3cd-cc6bf7fd95a2'
 '5050eefe-2392-4d1

In [577]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_deduplication_phase1.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_deduplication_phase1.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_deduplication_phase1.csv')

# Define datasets and their names and titles
datasets = [data1, data2, data3]
dataset_names = ['customer_demographics', 'customer_transactions', 'social_media_interactions']
dataset_titles = ['Customer Demographics', 'Customer Transactions', 'Social Media Interactions']
unique_id_columns = ['CustomerID', 'TransactionID', 'InteractionID']

# Select the dataset for search (in this case, data1)
df = data2

# Search for multiple items in one column
column_search = 'TransactionID'
search_items = [
    '5f93f1dc-0363-4e69-979b-f5d58fd42658',
 '7202af3a-448f-4970-9674-40f736cfdfde',
 '185ee140-348a-4461-bd40-07a7e8535815',
 '27fb25aa-f148-423f-967c-4d1764e26b29',
 'f39c8b1d-595a-4656-8108-25981bf3fa14',
 '652636b6-5c72-44e1-aefb-6adce412ee75',
 'eb26a4f4-5af2-4e25-b1f2-fd0de5583f7b',
 '7fbe5cc4-a35e-4e8e-89d7-7b439659b9f6',
 '5fca256b-cd5a-48fa-8b07-37f39adde8c9',
 'dc9453eb-3922-4a5f-91c0-ccb226ca91f7',
 '2e0b66a9-af5d-4cf9-bb60-55ab2932edee',
 '022cfd97-a8e8-4e50-8700-2720d52bfd75',
 '62186b07-9cda-44c5-bf76-bdb0f97aaf39',
 'c751a808-ed8d-44d0-8c52-8253ad564c9a',
 '6c2c5c2b-7601-41dc-9f56-4009de6f3cea'
]

# Filter the dataframe by the records in the search items
record_matches = df.loc[df[column_search].isin(search_items)]

# Group the results by the 'CustomerID' column
grouped_results = record_matches.groupby(column_search)

# Display grouped search results with clear formatting
print(f"Search results grouped by {column_search}:")

for customer_id, group in grouped_results:
    print(f"\nCustomerID: {customer_id}")
    print(group)  # Display the full group for this CustomerID


Search results grouped by TransactionID:

CustomerID: 022cfd97-a8e8-4e50-8700-2720d52bfd75
                                CustomerID                         TransactionID TransactionDate  Amount  ProductCategory PaymentMethod
2118  778da54b-8f96-4609-a21b-bba3d7f071ea  022cfd97-a8e8-4e50-8700-2720d52bfd75      2022-08-23  586.87  Health & Beauty        PayPal
3000  778da54b-8f96-4609-a21b-bba3d7f071ea  022cfd97-a8e8-4e50-8700-2720d52bfd75      2022-08-23    Free  Health & Beauty        PayPal

CustomerID: 185ee140-348a-4461-bd40-07a7e8535815
                                CustomerID                         TransactionID TransactionDate  Amount  ProductCategory PaymentMethod
413   6a84f1a7-f636-4300-bd8d-c7f888460829  185ee140-348a-4461-bd40-07a7e8535815      2023-10-09  953.86  Health & Beauty        PayPal
3013  6a84f1a7-f636-4300-bd8d-c7f888460829  185ee140-348a-4461-bd40-07a7e8535815      2023-10-09       0  Health & Beauty        PayPal

CustomerID: 27fb25aa-f148-423f-967c-4d1764

# b. Identify columns with missing values and decide how to handle them.

In [579]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_cleaned.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_cleaned.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_cleaned.csv')

# Define datasets and dataset_names
datasets = [data1, data2, data3] #replace
dataset_names = ['customer_demographics','customer_transactions','social_media_interactions']
dataset_titles = ['Customer Demographics','Customer Transactions','Social Media Interactions']

print('############################################################################################################################')

# Check for number of columns and rows for each dataset
print("\nAfter deduplication and cleaning:")
for df, dataset_name in zip(datasets, dataset_names):
    print(f"\n{dataset_name} has {df.shape[1]} columns and {df.shape[0]} rows.")
    # Check for data types
    print(f"Data types for each column:\n{df.dtypes}")


for df, dataset_name in zip(datasets, dataset_names):
    print(f"\n {dataset_name} statistics: \n{df.describe()}")

print('############################################################################################################################')
# Function to check for missing values and print only columns with missing values
def check_missing_values(datasets, dataset_titles):
    """
    Checks for missing values in the datasets and prints the variables with missing values.

    Parameters:
    datasets (list of pd.DataFrame): List of dataframes to check.
    dataset_names (list of str): List of dataset names corresponding to the dataframes.
    """
    for df, dataset_title in zip(datasets, dataset_titles):
        # print(f"\nMissing values for {dataset_name}:")
        missing_values = df.isnull().sum()  # Count missing values in each column
        missing_values = missing_values[missing_values > 0]  # Filter only those with missing values
        
        if not missing_values.empty:
            print(f"\nVariables with missing values in {dataset_title}:")
            for column, count in missing_values.items():
                print(f"{column}: {count} missing values")
        else:
            print(f"\nNo missing values found in {dataset_title}.")
            
# Call the function to check for and print columns with missing values
check_missing_values(datasets, dataset_titles)

print('############################################################################################################################')
# Calculate missingness for each column
def missingness_rate(datasets, dataset_titles):
    for df, dataset_title in zip(datasets, dataset_titles):
        missingness = df.isnull().sum() / len(df) * 100
        # Display missingness percentage for each column
        print(f"\nMissingness rate per column in {dataset_title}:\n{missingness}")

missingness_rate(datasets, dataset_titles)

############################################################################################################################

After deduplication and cleaning:

customer_demographics has 6 columns and 3000 rows.
Data types for each column:
CustomerID      object
Age            float64
Gender          object
Location        object
IncomeLevel     object
SignupDate      object
dtype: object

customer_transactions has 6 columns and 2997 rows.
Data types for each column:
CustomerID          object
TransactionID       object
TransactionDate     object
Amount             float64
ProductCategory     object
PaymentMethod       object
dtype: object

social_media_interactions has 6 columns and 2993 rows.
Data types for each column:
CustomerID         object
InteractionID      object
InteractionDate    object
Platform           object
InteractionType    object
Sentiment          object
dtype: object

 customer_demographics statistics: 
               Age
count  2695.000000
mean     45.390353
std 

In [580]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_cleaned.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_cleaned.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_cleaned.csv')

# Define datasets and dataset_names
datasets = [data1.copy(), data2.copy(), data3.copy()] #replace
dataset_names = ['customer_demographics','customer_transactions','social_media_interactions']
dataset_titles = ['Customer Demographics','Customer Transactions','Social Media Interactions']


# Specify columns to search for missing values in each dataset
columns_to_search = {
    'customer_demographics': ['CustomerID','Age', 'Gender','Location','IncomeLevel','SignupDate'],  
    'customer_transactions': ['CustomerID', 'TransactionID','TransactionDate', 'Amount','ProductCategory','PaymentMethod'],  
    'social_media_interactions': ['CustomerID', 'InteractionID','InteractionDate','Platform','InteractionType','Sentiment']
}

def search_null_columns(datasets, dataset_names, columns_to_search):
    for dataset, dataset_name in zip(datasets, dataset_names):
        print('\n*************************************************************************************************')
        print(f"Processing dataset: {dataset_name}")
        print('*************************************************************************************************')
        for column in columns_to_search[dataset_name]:
            # Search for records where the specified column is missing
            record_match = dataset.loc[dataset[column].isnull()].copy()  # Explicitly create a copy
            print('-----------------------------------------------------------------------------------------------')
            if record_match.empty:
                print(f"No missing values found in the column '{column}' in the dataset '{dataset_name}'.")
                continue

            # Count the number of missing values in each row for the matched records
            record_match['missing_count'] = record_match.isnull().sum(axis=1)

            # Display the search results along with the missing value count for each record
            print(f"Search results for records that have missing values under '{column}':")
            # print(record_match[[column, 'missing_count']])

            # Find the minimum and maximum number of missing values
            min_missing = record_match['missing_count'].min()
            max_missing = record_match['missing_count'].max()
            print(f"\nMinimum number of missing values in records: {min_missing}")
            print(f"Maximum number of missing values in records: {max_missing}")

            # Group by the number of missing values and count how many rows have the same number of missing values
            missing_value_summary = record_match['missing_count'].value_counts().sort_index()

            # Convert to DataFrame and rename the columns
            missing_value_summary_df = missing_value_summary.reset_index()
            missing_value_summary_df.columns = ['Number of Missing Values', 'Count of Record(s)']

            # Display the grouped and sorted number of missing values and the corresponding count of records
            print("\nSummary of records by the number of missing values:")
            print(missing_value_summary_df)

# Call the function with multiple datasets
search_null_columns(datasets, dataset_names, columns_to_search)


*************************************************************************************************
Processing dataset: customer_demographics
*************************************************************************************************
-----------------------------------------------------------------------------------------------
No missing values found in the column 'CustomerID' in the dataset 'customer_demographics'.
-----------------------------------------------------------------------------------------------
Search results for records that have missing values under 'Age':

Minimum number of missing values in records: 1
Maximum number of missing values in records: 2

Summary of records by the number of missing values:
   Number of Missing Values  Count of Record(s)
0                         1                 269
1                         2                  36
-----------------------------------------------------------------------------------------------
No missing values found i

# Delete records that have more than 1 missing value.

In [582]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_cleaned.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_cleaned.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_cleaned.csv')

# Define datasets and dataset_names
datasets = [data1.copy(), data2.copy(), data3.copy()]  # replace
dataset_names = ['customer_demographics', 'customer_transactions', 'social_media_interactions']
dataset_titles = ['Customer Demographics', 'Customer Transactions', 'Social Media Interactions']

# Specify columns to search for missing values in each dataset
columns_to_search = {
    'customer_demographics': ['CustomerID', 'Age', 'Gender', 'Location', 'IncomeLevel', 'SignupDate'],
    'customer_transactions': ['CustomerID', 'TransactionID', 'TransactionDate', 'Amount', 'ProductCategory', 'PaymentMethod'],
    'social_media_interactions': ['CustomerID', 'InteractionID', 'InteractionDate', 'Platform', 'InteractionType', 'Sentiment']
}

# Function to delete records with 2 or more missing values and save the cleaned datasets
def delete_missing_records(datasets, dataset_names, columns_to_search):
    for dataset, dataset_name in zip(datasets, dataset_names):
        print(f"\nProcessing dataset: {dataset_name}")
        
        # Select columns to search for missing values
        columns = columns_to_search[dataset_name]
        
        # Count the number of missing values across the specified columns in each record
        dataset['missing_count'] = dataset[columns].isnull().sum(axis=1)
        
        # Filter out records with 2 or more missing values
        cleaned_dataset = dataset[dataset['missing_count'] < 2].drop(columns=['missing_count'])
        
        # Save the cleaned dataset to a CSV file with "_deleted_missing_records" appended to the filename
        cleaned_filename = f'/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/{dataset_name}_deleted_missing_records.csv'
        cleaned_dataset.to_csv(cleaned_filename, index=False)
        
        print(f"Saved cleaned dataset as: {cleaned_filename}")
        print(f"Number of records removed: {len(dataset) - len(cleaned_dataset)}")
        print(f"Number of remaining records: {len(cleaned_dataset)}\n")

# Call the function with the datasets and corresponding names
delete_missing_records(datasets, dataset_names, columns_to_search)


Processing dataset: customer_demographics
Saved cleaned dataset as: /Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_deleted_missing_records.csv
Number of records removed: 36
Number of remaining records: 2964


Processing dataset: customer_transactions
Saved cleaned dataset as: /Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_deleted_missing_records.csv
Number of records removed: 26
Number of remaining records: 2971


Processing dataset: social_media_interactions
Saved cleaned dataset as: /Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_deleted_missing_records.csv
Number of records removed: 27
Number of remaining records: 2966



# Perform mean imputation for missing Age values

In [584]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_deleted_missing_records.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_deleted_missing_records.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_deleted_missing_records.csv')

# Define datasets and dataset_names
datasets = [data1.copy(), data2.copy(), data3.copy()]  # replace
dataset_names = ['customer_demographics', 'customer_transactions', 'social_media_interactions']
dataset_titles = ['Customer Demographics', 'Customer Transactions', 'Social Media Interactions']

# Function to check for missing values and print only columns with missing values
def check_missing_values(datasets, dataset_titles):
    """
    Checks for missing values in the datasets and prints the variables with missing values.

    Parameters:
    datasets (list of pd.DataFrame): List of dataframes to check.
    dataset_names (list of str): List of dataset names corresponding to the dataframes.
    """
    for df, dataset_title in zip(datasets, dataset_titles):
        # print(f"\nMissing values for {dataset_name}:")
        missing_values = df.isnull().sum()  # Count missing values in each column
        missing_values = missing_values[missing_values > 0]  # Filter only those with missing values
        
        if not missing_values.empty:
            print(f"\nVariables with missing values in {dataset_title}:")
            for column, count in missing_values.items():
                print(f"{column}: {count} missing values")
        else:
            print(f"\nNo missing values found in {dataset_title}.")
            
# Call the function to check for and print columns with missing values
check_missing_values(datasets, dataset_titles)


Variables with missing values in Customer Demographics:
Age: 269 missing values
IncomeLevel: 248 missing values

Variables with missing values in Customer Transactions:
Amount: 255 missing values
ProductCategory: 254 missing values

Variables with missing values in Social Media Interactions:
Platform: 262 missing values
Sentiment: 280 missing values


In [585]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_deleted_missing_records.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_deleted_missing_records.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_deleted_missing_records.csv')

# Define datasets and dataset names
datasets = [data1.copy(), data2.copy(), data3.copy()]
dataset_names = ['customer_demographics', 'customer_transactions', 'social_media_interactions']
dataset_titles = ['Customer Demographics', 'Customer Transactions', 'Social Media Interactions']

# Columns with missing values to be imputed
columns_to_impute = {
    'customer_demographics': {'numeric': ['Age'], 'categorical': ['IncomeLevel']},
    'customer_transactions': {'numeric': ['Amount'], 'categorical': ['ProductCategory']},
    'social_media_interactions': {'categorical': ['Platform', 'Sentiment']}
}

# Function to perform mean and mode imputation using SimpleImputer
def impute_missing_values_sklearn(datasets, dataset_names, columns_to_impute):
    for dataset, dataset_name in zip(datasets, dataset_names):
        print(f'\nImputing missing values for {dataset_name} dataset...')

        # Impute numeric columns with the mean
        if 'numeric' in columns_to_impute[dataset_name]:
            numeric_columns = columns_to_impute[dataset_name]['numeric']
            imputer_mean = SimpleImputer(strategy='mean')
            dataset[numeric_columns] = imputer_mean.fit_transform(dataset[numeric_columns])
            print(f"Mean imputation applied for columns: {numeric_columns}")

        # Impute categorical columns with the most frequent (mode)
        if 'categorical' in columns_to_impute[dataset_name]:
            categorical_columns = columns_to_impute[dataset_name]['categorical']
            imputer_mode = SimpleImputer(strategy='most_frequent')
            dataset[categorical_columns] = imputer_mode.fit_transform(dataset[categorical_columns])
            print(f"Mode imputation applied for columns: {categorical_columns}")

        # Save the imputed dataset to a new CSV file
        new_file_name = f'/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/{dataset_name}_imputed_sklearn.csv'
        dataset.to_csv(new_file_name, index=False)
        print(f"Imputed dataset saved as: {new_file_name}")

# Call the function to perform imputation using scikit-learn
impute_missing_values_sklearn(datasets, dataset_names, columns_to_impute)


Imputing missing values for customer_demographics dataset...
Mean imputation applied for columns: ['Age']
Mode imputation applied for columns: ['IncomeLevel']
Imputed dataset saved as: /Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_imputed_sklearn.csv

Imputing missing values for customer_transactions dataset...
Mean imputation applied for columns: ['Amount']
Mode imputation applied for columns: ['ProductCategory']
Imputed dataset saved as: /Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_imputed_sklearn.csv

Imputing missing values for social_media_interactions dataset...
Mode imputation applied for columns: ['Platform', 'Sentiment']
Imputed dataset saved as: /Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_imputed_sklearn.csv


In [586]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_imputed_sklearn.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_imputed_sklearn.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_imputed_sklearn.csv')

# Define datasets and dataset_names
datasets = [data1.copy(), data2.copy(), data3.copy()]  # replace
dataset_names = ['customer_demographics', 'customer_transactions', 'social_media_interactions']
dataset_titles = ['Customer Demographics', 'Customer Transactions', 'Social Media Interactions']

# Function to check for missing values and print only columns with missing values
def check_missing_values(datasets, dataset_titles):
    """
    Checks for missing values in the datasets and prints the variables with missing values.

    Parameters:
    datasets (list of pd.DataFrame): List of dataframes to check.
    dataset_names (list of str): List of dataset names corresponding to the dataframes.
    """
    for df, dataset_title in zip(datasets, dataset_titles):
        # print(f"\nMissing values for {dataset_name}:")
        missing_values = df.isnull().sum()  # Count missing values in each column
        missing_values = missing_values[missing_values > 0]  # Filter only those with missing values
        
        if not missing_values.empty:
            print(f"\nVariables with missing values in {dataset_title}:")
            for column, count in missing_values.items():
                print(f"{column}: {count} missing values")
        else:
            print(f"\nNo missing values found in {dataset_title}.")
            
# Call the function to check for and print columns with missing values
check_missing_values(datasets, dataset_titles)


No missing values found in Customer Demographics.

No missing values found in Customer Transactions.

No missing values found in Social Media Interactions.


# c. Ensure columns have the appropriate data types.

In [588]:
def check_variable_types(datasets, dataset_titles):
    """
    Checks and prints the data types of each column in the datasets.

    Parameters:
    datasets (list of pd.DataFrame): List of dataframes to check.
    dataset_names (list of str): List of dataset names corresponding to the dataframes.
    """
    for df, dataset_title in zip(datasets, dataset_titles):
        print(f"\nData types for {dataset_title} variables:")
        print(df.dtypes)

# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_imputed_sklearn.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_imputed_sklearn.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_imputed_sklearn.csv')

# Define datasets and dataset_names
datasets = [data1.copy(), data2.copy(), data3.copy()]  # replace
dataset_names = ['customer_demographics', 'customer_transactions', 'social_media_interactions']
dataset_titles = ['Customer Demographics', 'Customer Transactions', 'Social Media Interactions']


# Call the function to check variable types
check_variable_types(datasets, dataset_titles)


Data types for Customer Demographics variables:
CustomerID      object
Age            float64
Gender          object
Location        object
IncomeLevel     object
SignupDate      object
dtype: object

Data types for Customer Transactions variables:
CustomerID          object
TransactionID       object
TransactionDate     object
Amount             float64
ProductCategory     object
PaymentMethod       object
dtype: object

Data types for Social Media Interactions variables:
CustomerID         object
InteractionID      object
InteractionDate    object
Platform           object
InteractionType    object
Sentiment          object
dtype: object


In [589]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_imputed_sklearn.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_imputed_sklearn.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_imputed_sklearn.csv')

# Define datasets and dataset_names
datasets = [data1.copy(), data2.copy(), data3.copy()]  # replace
dataset_names = ['customer_demographics', 'customer_transactions', 'social_media_interactions']
dataset_titles = ['Customer Demographics', 'Customer Transactions', 'Social Media Interactions']


from datetime import datetime

# Define common date formats
common_date_formats = [
    "%Y-%m-%d",  # 2024-11-01
    "%d/%m/%Y",  # 01/11/2024
    "%m/%d/%Y",  # 11/01/2024
    "%d-%m-%Y",  # 01-11-2024
    "%m-%d-%Y",  # 11-01-2024
    "%Y/%m/%d",  # 2024/11/01
    "%d.%m.%Y",  # 01.11.2024
    "%m.%d.%Y",  # 11.01.2024
]

# Function to check current date format
def detect_date_format(date_series):
    for date_format in common_date_formats:
        try:
            # Try parsing the first non-null date in the series with each format
            test_date = date_series.dropna().iloc[0]
            datetime.strptime(test_date, date_format)
            return date_format
        except (ValueError, IndexError):
            continue
    return None  # Return None if no format matched

# Call function
detected_format = detect_date_format(data3['InteractionDate'])
if detected_format:
    print(f"Detected date format for SignupDate: {detected_format}")
else:
    print("Could not detect a consistent date format.")

Detected date format for SignupDate: %d/%m/%Y


In [610]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_demographics_imputed_sklearn.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/customer_transactions_imputed_sklearn.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/social_media_interactions_imputed_sklearn.csv')

# Parse the SignupDate in customer demographics with the identified format
data1['SignupDate'] = pd.to_datetime(data1['SignupDate'], format='%d/%m/%Y', errors='coerce')

# Parse the TransactionDate in customer transactions with the identified format
data2['TransactionDate'] = pd.to_datetime(data2['TransactionDate'], format='%d/%m/%Y', errors='coerce')

# Parse the InteractionDate in social media interactions with the identified format
data3['InteractionDate'] = pd.to_datetime(data3['InteractionDate'], format='%d/%m/%Y', errors='coerce')

# Ensure that 'Age' is numeric and round down any float values before converting to integers
data1['Age'] = pd.to_numeric(data1['Age'], errors='coerce').apply(np.floor).astype('Int64')

# Format the 'Amount' column in customer transactions as float and round it to 2 decimal places
data2['Amount'] = pd.to_numeric(data2['Amount'], errors='coerce').round(2)

# Define datasets and dataset_names
datasets = [data1.copy(), data2.copy(), data3.copy()]
dataset_names = ['customer_demographics', 'customer_transactions', 'social_media_interactions']

# Save the standardized dataset with '_standardized' suffix
for dataset, name in zip(datasets, dataset_names):
    dataset.to_csv(f'/Users/fritzcabalhin/Desktop/2. Machine Learning/Codes/{name}_standardized.csv', index=False)

# Confirm the changes
print(data1.dtypes)  # Check that 'Age' is now an integer
print(data2.dtypes)  # Check that 'Amount' is now a float
print(data3.dtypes)  # Check 'InteractionDate' has been parsed correctly

CustomerID             object
Age                     Int64
Gender                 object
Location               object
IncomeLevel            object
SignupDate     datetime64[ns]
dtype: object
CustomerID                 object
TransactionID              object
TransactionDate    datetime64[ns]
Amount                    float64
ProductCategory            object
PaymentMethod              object
dtype: object
CustomerID                 object
InteractionID              object
InteractionDate    datetime64[ns]
Platform                   object
InteractionType            object
Sentiment                  object
dtype: object
